# Loopcom Yiddish Whisper fine-tune — Kaggle free-GPU run

Pushed and started by `kaggle-run.ts kernel-push`. The attached dataset
(`kernel-metadata.json` -> `dataset_sources`) is `build-dataset.ts`'s output
folder, self-contained: `clips/`, `train.jsonl`, `eval.jsonl`,
`manifest.json`, plus copies of `train.py`, `baseline.py` and
`requirements-kaggle.txt` that `kaggle-run.ts dataset-push` staged alongside
the audio.

This notebook installs deps (torch is already present on the Kaggle GPU
image — see `requirements-kaggle.txt`'s own comment for why it is never
reinstalled here), runs `train.py` with the 16GB-card defaults (batch 4,
grad-accum 8, fp16/bf16 auto-detected, LoRA by default), and prints
`report.json` at the end. `kaggle-run.ts download` pulls `/kaggle/working/`
back afterwards.

In [ ]:
import glob, os, shutil, subprocess, sys

# Kaggle mounts every dataset in kernel-metadata.json's dataset_sources under
# /kaggle/input/<slug>/. There should be exactly one attached here.
candidates = [p for p in glob.glob('/kaggle/input/*') if os.path.isdir(p)]
assert candidates, 'no dataset attached under /kaggle/input — check kernel-metadata.json dataset_sources'
DATASET_DIR = candidates[0]
print('dataset dir:', DATASET_DIR)
print('contents:', sorted(os.listdir(DATASET_DIR))[:20])

os.chdir('/kaggle/working')
for name in ('train.py', 'baseline.py', 'requirements-kaggle.txt'):
    src = os.path.join(DATASET_DIR, name)
    assert os.path.exists(src), f'{name} missing from the dataset — dataset-push must copy it in'
    shutil.copy(src, name)


In [ ]:
# torch/torchaudio are already on the Kaggle GPU image — requirements-kaggle.txt
# deliberately omits them (see that file's comment).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())


In [ ]:
# 16GB-card defaults live IN train.py (batch 4, grad-accum 8, max-steps 2000,
# eval every 250 steps, bf16-if-supported else fp16) — nothing to override
# here for a T4/P100 session. Pass --max-steps/--batch-size to shorten a run
# that would not fit in Kaggle's 12h session limit.
subprocess.run(
    [sys.executable, 'train.py', '--dataset', DATASET_DIR, '--output-dir', '/kaggle/working/out'],
    check=True,
)


In [ ]:
import json
with open('report.json') as f:
    report = json.load(f)
print(json.dumps(report, indent=2))
